In [ ]:
import pandas as pd
data = pd.read_csv("dataset/dataset_for_router.csv", index_col=0)

result = pd.read_csv("dataset/results_for_router_matching_based.csv", index_col=0)

true_labels = result["label"].tolist()
predicted_labels = result["predicted"].tolist()
from sklearn.metrics import classification_report
report = classification_report(true_labels, predicted_labels)
print(report)

              precision    recall  f1-score   support

   documents       0.53      0.92      0.67      1000
extract_laws       0.70      0.18      0.28      1000

    accuracy                           0.55      2000
   macro avg       0.61      0.55      0.48      2000
weighted avg       0.61      0.55      0.48      2000



In [2]:
true_prediction = sum([1 for i in range(len(true_labels)) if true_labels[i] == predicted_labels[i]])
accuracy = true_prediction / len(true_labels)
print("Accuracy:", accuracy)

Accuracy: 0.5495


Classifier based router

In [85]:
from agent import router_classifier_based
data = pd.read_csv("dataset/dataset_for_router.csv", index_col=0)
predicted_labels = []
for query in data["query"].tolist():
    predicted_label = router_classifier_based({"query": query, "uploaded_files": ["Cristiano Ronaldo.pdf"]})
    predicted_labels.append(predicted_label["route"])
true_labels = data["label"].tolist()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): documents
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classifier): extract_laws
Router chọn (bằng classi

In [88]:
from sklearn.metrics import classification_report
report = classification_report(true_labels, predicted_labels)
print(report)

              precision    recall  f1-score   support

   documents       0.96      0.99      0.97      1000
extract_laws       0.99      0.96      0.97      1000

    accuracy                           0.97      2000
   macro avg       0.97      0.97      0.97      2000
weighted avg       0.97      0.97      0.97      2000



Prompt based router

In [ ]:
from llm import router
data = pd.read_csv("dataset/dataset_for_router.csv", index_col=0)
predicted_labels = []
true_labels = data["label"].tolist()

for query in data["query"].tolist():
    predicted_label = router({"query": query, "uploaded_files": ["Cristiano Ronaldo.pdf"]})
    predicted_labels.append(predicted_label["route"])
    print(f"Predicted label: {predicted_label['route']}, True label: {true_labels[i]}")

Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: extract_laws
Predicted label: extract_laws, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router chọn: documents
Predicted label: documents, True label: extract_laws
Router

In [38]:
report = classification_report(true_labels, predicted_labels)
print(report)

              precision    recall  f1-score   support

   documents       0.52      0.95      0.67      1000
extract_laws       0.73      0.13      0.22      1000

    accuracy                           0.54      2000
   macro avg       0.62      0.54      0.45      2000
weighted avg       0.62      0.54      0.45      2000



Router embedding similarity

In [58]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/taidng/UIT-ViQuAD2.0/" + splits["test"])

In [6]:
from index import chunk_text_by_tokens, split_into_sentences, get_embedder
import faiss
import numpy as np

In [61]:
df = df.sample(n=1000, random_state=42)
queries = df["question"][:1000].tolist()
contexts = df["context"][:1000].tolist()

data = pd.DataFrame({"query": queries, "docs_context": contexts, "label": "documents"})
data.to_csv("dataset/dataset_for_embedding_similarity.csv")

In [63]:
data_laws = pd.read_csv("dataset/sent_truncated_dvc_train.csv", index_col=0, encoding="utf-16")

In [64]:
law_contexts = contexts
queries = data_laws["noi_dung_hoi"][:1000].tolist()

data_laws_final = pd.DataFrame({"query": queries, "docs_context": law_contexts, "label": "extract_laws"})

In [65]:
print(len(law_contexts), len(queries))

1000 1000


In [66]:
data_final = pd.concat([data, data_laws_final], ignore_index=True)

In [70]:
data_final.to_csv("dataset/dataset_for_router_embedding_similarity.csv", index=False)

In [71]:
def get_faiss_index_for_uploads(input_dict: dict, max_tokens: int = 20, stride: int = 5):
    """
    Build faiss index for a set of uploaded files, with caching.
    """
    sentences = []
    text = input_dict.get("docs_context", "")
    page_chunks = chunk_text_by_tokens(text, max_tokens=max_tokens, stride=stride)
    if not page_chunks:
        sents = split_into_sentences(text)
        page_chunks = chunk_text_by_tokens(" ".join(sents), max_tokens=max_tokens, stride=stride)

    sentences.extend(page_chunks)

    emb = get_embedder()
    embeddings = emb.encode(sentences, convert_to_numpy=True, show_progress_bar=False)
    embeddings = np.asarray(embeddings, dtype=np.float32)

    dim = embeddings.shape[1]
    idx = faiss.IndexFlatL2(dim)
    idx.add(embeddings)
    return idx, sentences

In [82]:
from index import get_laws_index_and_json, get_embedder
from llm import distance_to_sim

In [86]:
def router_embedding_similarity(input_dict:dict, top_k:int = 1) -> dict:
    """Route the query by comparing embedding-similarity against the laws index and uploaded-files index."""
    laws_index, laws = get_laws_index_and_json()
    docs_context = input_dict.get("docs_context", "")
    if docs_context:
        keywords = ["tóm tắt", "tổng hợp", "tóm tắt văn bản", "tóm tắt nội dung", "ngắn gọn nội dung"]
        if any(keyword in input_dict.get("query", "").lower() for keyword in keywords):
            route = "documents"
        else:
            docs_index, _ = get_faiss_index_for_uploads({"docs_context": docs_context})
            emb = get_embedder()
            query_embedding = emb.encode([input_dict.get("query","")])
            D_laws, I_laws = laws_index.search(query_embedding, k = top_k)
            D_docs, I_docs = docs_index.search(query_embedding, k = top_k)
            D_docs_avg = np.mean(D_docs)
            D_laws_avg = np.mean(D_laws)
            D_docs_sim = distance_to_sim(docs_index, D_docs_avg)
            D_laws_sim = distance_to_sim(laws_index, D_laws_avg)
            if D_docs_sim > D_laws_sim:
                route = "documents"
            elif abs(D_docs_sim - D_laws_sim) < 0.01:
                route = "documents"
            else:
                route = "extract_laws"
    else:
        route = "extract_laws"
    return {"route": route, "uploaded_files": input_dict.get("uploaded_files",[]), "query": input_dict.get("query","")}

In [87]:
data_final = pd.read_csv("dataset/dataset_for_router_embedding_similarity.csv")
#shuffle the data
data_final = data_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [92]:
predicted_labels = []
true_labels = data_final["label"].tolist()
i =0
import tqdm 
for row in tqdm.tqdm(data_final.itertuples()):
    predicted_label = router_embedding_similarity({"query": row.query, "docs_context": row.docs_context})
    predicted_labels.append(predicted_label["route"])

2000it [01:01, 32.63it/s]


In [93]:
from sklearn.metrics import classification_report
report = classification_report(true_labels, predicted_labels)
print(report)

              precision    recall  f1-score   support

   documents       1.00      0.97      0.98      1000
extract_laws       0.97      1.00      0.98      1000

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000



Retrain the classifier based router with new data

In [46]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
docs_df = pd.read_parquet("hf://datasets/taidng/UIT-ViQuAD2.0/" + splits["train"])

In [47]:
docs_df.shape

(28454, 8)

In [48]:
docs_df.columns

Index(['id', 'uit_id', 'title', 'context', 'question', 'answers',
       'is_impossible', 'plausible_answers'],
      dtype='object')

In [49]:
docs_queries = docs_df["question"][:8000].tolist()

In [50]:
summary_queries = ["Tóm tắt nội dung văn bản nhập lên", "Ngắn gọn nội dung văn bản"]*1000

In [51]:
docs_queries.extend(summary_queries)

In [53]:
data_docs = pd.DataFrame({"query": docs_queries, "label": "documents"})

In [54]:
data_docs.shape

(10000, 2)

In [55]:
data_laws = pd.read_csv("dataset/sent_truncated_zac_train.csv", index_col=0, encoding="utf-16")

In [56]:
laws_queries = data_laws["noi_dung_hoi"][:20000].tolist()

In [57]:
len(laws_queries)

8665

In [58]:
laws_data = pd.DataFrame({"query": laws_queries, "label": "extract_laws"})

In [59]:
laws_gpt = pd.read_excel("path_classifier.xlsx", engine="openpyxl")

In [60]:
laws_gpt = laws_gpt[laws_gpt["label"]=="extract_laws"]

In [61]:
laws_final = pd.concat([laws_data, laws_gpt])

In [62]:
laws_final.shape

(8916, 2)

In [63]:
data_final = pd.concat([data_docs, laws_final], ignore_index=True)

In [65]:
data_final.shape

(18916, 2)

In [64]:
data_final.to_csv("dataset/dataset_for_path_classifier_training.csv", index=False)

In [77]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

# Expose an embedder and label map for external scripts (e.g., test.py)
embedder = SentenceTransformer("intfloat/multilingual-e5-large")

# simple label mapping used by the classifier
label_map = {"documents": 0, "extract_laws": 1}

def map_label(label):
    return label_map[label]

def embedding_sentence(sentence):
    embedding = embedder.encode([sentence])
    return embedding[0]

class PathClassifier(nn.Module):
    def __init__(self,input_size = 1024, num_classes = 2):
        super(PathClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 512)
        self.fc2 = nn.Linear(512, 2)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        # return raw logits (no activation) so CrossEntropyLoss can be applied
        x = self.fc2(x)
        return x

def train_model(model, x_train, y_train, x_test, y_test, epochs=20, learning_rate=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        print(f"----------- Epoch {epoch+1}/{epochs} -----------")
        model.train()
        output = model(x_train)
        loss = loss_fn(output, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            test_output = model(x_test)
            test_loss = loss_fn(test_output, y_test)
            _, predicted = torch.max(test_output, 1)
            accuracy = (predicted == y_test).sum().item() / len(y_test)
        print(f"Train Loss: {loss.item():.4f} | Test Loss: {test_loss.item():.4f} | Test Accuracy: {accuracy*100:.2f}%")

In [68]:
data  = pd.read_csv("dataset/dataset_for_path_classifier_training.csv")
data  = data.sample(frac=1, random_state = 7).reset_index(drop=True)

#apply embedding to each query with progress bar
data["embedding"] = data["query"].apply(embedding_sentence)
data["label"] = data["label"].apply(map_label)

In [84]:
train_size = int(0.8*len(data))
test_size = len(data) - train_size

train_data, test_data = train_test_split(data, test_size=test_size, stratify=data['label'], random_state=8)
y_train = torch.tensor(train_data["label"].to_numpy(), dtype=torch.long)
x_train = torch.tensor(np.stack(train_data["embedding"].to_numpy()), dtype=torch.float32)
y_test = torch.tensor(test_data["label"].to_numpy(), dtype=torch.long)
x_test = torch.tensor(np.stack(test_data["embedding"].to_numpy()), dtype=torch.float32)

model = PathClassifier()
train_model(model, x_train, y_train, x_test, y_test, epochs=80, learning_rate=0.0001)
torch.save(model.state_dict(), "path_classifier_model.pth")

----------- Epoch 1/80 -----------
Train Loss: 0.6936 | Test Loss: 0.6921 | Test Accuracy: 62.21%
----------- Epoch 2/80 -----------
Train Loss: 0.6922 | Test Loss: 0.6906 | Test Accuracy: 58.11%
----------- Epoch 3/80 -----------
Train Loss: 0.6907 | Test Loss: 0.6891 | Test Accuracy: 55.95%
----------- Epoch 4/80 -----------
Train Loss: 0.6893 | Test Loss: 0.6877 | Test Accuracy: 55.23%
----------- Epoch 5/80 -----------
Train Loss: 0.6879 | Test Loss: 0.6863 | Test Accuracy: 55.15%
----------- Epoch 6/80 -----------
Train Loss: 0.6863 | Test Loss: 0.6849 | Test Accuracy: 55.02%
----------- Epoch 7/80 -----------
Train Loss: 0.6850 | Test Loss: 0.6835 | Test Accuracy: 55.34%
----------- Epoch 8/80 -----------
Train Loss: 0.6836 | Test Loss: 0.6820 | Test Accuracy: 55.58%
----------- Epoch 9/80 -----------
Train Loss: 0.6822 | Test Loss: 0.6806 | Test Accuracy: 56.18%
----------- Epoch 10/80 -----------
Train Loss: 0.6809 | Test Loss: 0.6792 | Test Accuracy: 56.77%
----------- Epoch 1